# C1 EfficientNetV2-S Baseline
Run these cells in order with a GPU runtime. Select the dataset ZIP from your laptop when prompted. The test split is not used during training or checkpoint selection. Colab storage is temporary, so download the result ZIP before ending the runtime.

In [ ]:
import torch
import torchvision

print("=== C1 COLAB GPU CHECK ===")
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing"
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))

In [ ]:
import gc
from pathlib import Path
from google.colab import files

print("Select C1_colab_bundle.zip from your laptop.")
uploaded = files.upload()
expected_name = "C1_colab_bundle.zip"
assert expected_name in uploaded, f"Expected {expected_name}; received: {list(uploaded)}"
archive = Path("/content") / expected_name
assert archive.is_file() and archive.stat().st_size > 0
print("Uploaded:", archive)
print("Size (GB):", round(archive.stat().st_size / (1024 ** 3), 3))
del uploaded
gc.collect()

In [ ]:
import shutil

workspace = Path("/content/J26-IT-359")
assert archive.is_file(), f"Bundle not found: {archive}"
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
shutil.unpack_archive(archive, workspace)
print("Extracted bundle to:", workspace)

In [ ]:
import csv
from collections import Counter

component = workspace / "components/c1-bird-recognition"
manifest = component / "data/reports/experiment_manifest_provisional.csv"
training_script = component / "src/train_efficientnetv2_baseline.py"
assert manifest.is_file()
assert training_script.is_file()
with manifest.open(encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file))
included = [row for row in rows if row["experiment_status"] == "include"]
counts = Counter(row["split"] for row in included)
assert counts == {"train": 3792, "val": 814, "test": 814}, counts
missing = [row["image_path"] for row in included if not (workspace / row["image_path"]).is_file()]
assert not missing, f"Missing included images: {len(missing)}"
print("Included split counts:", dict(counts))
print("Missing included images: 0")
print("BUNDLE VALIDATION: PASS")

The default batch size is 16. If Colab reports a CUDA out-of-memory error, change `--batch-size 16` to `--batch-size 8` and rerun this cell.

In [ ]:
import subprocess

output_dir = Path("/content/C1_Bird_Recognition/runs/baseline")
output_dir.mkdir(parents=True, exist_ok=True)
command = [
    "python", str(training_script),
    "--output-dir", str(output_dir),
    "--epochs", "10",
    "--batch-size", "16",
    "--num-workers", "2",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

Download the result archive immediately after training. Files under `/content` are deleted when the Colab runtime ends.

In [ ]:
required_results = [
    "config.json",
    "history.json",
    "best_model.pt",
    "best_validation_metrics.json",
]
missing_results = [name for name in required_results if not (output_dir / name).is_file()]
assert not missing_results, f"Missing training results: {missing_results}"
result_archive = shutil.make_archive(
    "/content/C1_EfficientNetV2_Baseline_results",
    "zip",
    root_dir=output_dir,
)
print("Downloading:", result_archive)
files.download(result_archive)